# Achats de produits phytosanitaires par département (2013-2023)

## Introduction

Ce notebook constitue la partie **pesticides** du projet, qui cherche à relier les pratiques agricoles
à la qualité des cours d'eau. Avant de croiser les achats de pesticides avec les parcelles agricoles et
les indices de qualité de l'eau, il faut connaître la source : sa fiabilité, l'ordre de grandeur des
quantités, leur répartition sur le territoire et leur évolution dans le temps.

**Données.** Base nationale des ventes des distributeurs (BNV-D), publiée sur data.gouv.fr : quantités de
substances actives achetées (en kg), par département, année, substance, statut biocontrôle et classe de danger.
Le fichier est téléchargé automatiquement dans `data/raw/` s'il n'y est pas déjà.

**Questions.**
- Les données sont-elles complètes et cohérentes ?
- Où achète-t-on le plus de pesticides, et comment ces achats évoluent-ils ?
- Que deviennent les **10 molécules retenues par le projet** pour leur impact sur l'eau
  (`src/pesticides/constantes_pesticides.py`) : où sont-elles achetées, et lesquelles reculent ?

## Sommaire

0. [Configuration](#section-0)
1. [Chargement](#section-1)
2. [Qualité des données](#section-2)
3. [Préparation](#section-3)
4. [Vue d'ensemble : cartes et évolution annuelle](#section-4)
5. [Les 10 molécules du projet (dont cartes par molécule)](#section-5)
6. [Distributions](#section-6)
7. [Concentration géographique](#section-7)
8. [Tendances (dont cartes d'évolution)](#section-8)
9. [Conclusion](#section-9)

<a id="section-0"></a>

## 0. Configuration

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter

# Se placer à la racine du dépôt (le notebook est dans notebooks/exploration/)
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src").is_dir():
        os.chdir(parent)
        break

from src.pesticides.constantes_pesticides import MOLECULES
from src.pesticides.preparer_pesticides import VARIANTES_CUIVRE
from src.pesticides.recuperer_pesticides import recuperer_pesticides
from src.visualisation.calculer_gini import calculer_gini
from src.visualisation.tracer_carte import tracer_carte

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.1f}".format)

ANNEES = range(2013, 2024)
ANNEE_DEBUT_FIABLE = 2014  # 2013 est incomplète (voir section 2)
DEPARTEMENTS_EXCLUS = ["2A", "2B", "971", "972", "973", "974", "976"]  # même règle que preparer_pesticides
en_tonnes = FuncFormatter(lambda x, _: f"{x / 1e3:,.0f}")

<a id="section-1"></a>

## 1. Chargement

In [ ]:
chemin_csv = recuperer_pesticides()  # télécharge le CSV seulement s'il est absent de data/raw/
df = pd.read_csv(chemin_csv, dtype={"geocode_departement": str})  # garde "01" et "2A" en texte

print("Dimensions :", df.shape)
df.head()

<a id="section-2"></a>

## 2. Qualité des données

Une ligne = un département, une année (`date_mesure` au 1er janvier), une substance,
un statut biocontrôle et une classe de danger. `valeur` est la quantité achetée en kg de substance active.

In [ ]:
df.info()
pd.concat(
    [df.isna().sum().rename("valeurs_manquantes"), df.nunique().rename("modalites")],
    axis=1,
)

In [ ]:
df["date_mesure"] = pd.to_datetime(df["date_mesure"])
df["annee"] = df["date_mesure"].dt.year

print("Période :", df["date_mesure"].min().date(), "->", df["date_mesure"].max().date())
print("Valeurs négatives :", (df["valeur"] < 0).sum())
print("Valeurs nulles    :", (df["valeur"] == 0).sum())
print("Classes de danger :", sorted(df["danger_substance"].unique()))
print("Biocontrôle       :", sorted(df["biocontrole"].unique()))

display(df["valeur"].describe().to_frame())
display(
    df.groupby("annee")
      .agg(nb_lignes=("valeur", "size"), nb_departements=("geocode_departement", "nunique"))
      .T
)

In [ ]:
codes = sorted(df["geocode_departement"].unique())
print(len(codes), "codes départementaux, dont exclus de l'analyse :",
      [c for c in codes if c in DEPARTEMENTS_EXCLUS])

print("\nLes 10 plus grosses lignes :")
df.nlargest(10, "valeur")[["annee", "libelle_departement", "substance", "valeur"]]

**Plusieurs lignes par département, année et substance ?** On vérifie si elles s'expliquent
par le statut biocontrôle ou la classe de danger.

In [ ]:
analyse_doublons = (
    df.groupby(["geocode_departement", "annee", "substance"])
      .agg(
          nb_lignes=("valeur", "size"),
          nb_biocontrole=("biocontrole", "nunique"),
          nb_categories_danger=("danger_substance", "nunique"),
      )
)
groupes_multiples = analyse_doublons[analyse_doublons["nb_lignes"] > 1]

print("Doublons exacts sur les variables descriptives :",
      df.duplicated(subset=["date_mesure", "geocode_departement", "biocontrole",
                            "danger_substance", "substance"]).sum())
print("Groupes avec plusieurs lignes :", len(groupes_multiples))
print("  dont biocontrôle variable   :", (groupes_multiples["nb_biocontrole"] > 1).sum())
print("  dont danger variable        :", (groupes_multiples["nb_categories_danger"] > 1).sum())

# Exemple : pyrèthrines dans l'Ain en 2013
df.query("geocode_departement == '01' and annee == 2013 and substance == 'pyrethrines'")[
    ["annee", "substance", "biocontrole", "danger_substance", "valeur"]
]

**À retenir**
- Il n'y a ni valeur manquante ni doublon exact. Quand une substance a plusieurs lignes pour
  un même département et une même année, c'est **toujours** parce qu'elle est découpée en biocontrôle /
  hors biocontrôle (la classe de danger ne varie jamais). On peut donc **additionner** ces lignes.
- **2013 est incomplète** : environ 15 % de lignes en moins que les autres années et des quantités
  2 à 3 fois plus faibles (voir la section 4). On la garde dans les graphiques, mais les comparaisons
  de début et de fin de période partent de **2014**.

<a id="section-3"></a>

## 3. Préparation

- regroupement des 8 sels de cuivre sous le libellé `cuivre` (comme dans `preparer_pesticides`) ;
- exclusion de la Corse et des DOM, comme dans le reste du projet ;
- agrégation : une ligne par département, année, substance et classe de danger.

In [ ]:
df["substance"] = df["substance"].str.strip().replace(dict.fromkeys(VARIANTES_CUIVRE, "cuivre"))
df_metro = df[~df["geocode_departement"].isin(DEPARTEMENTS_EXCLUS)]

df_agrege = (
    df_metro.groupby(
        ["geocode_departement", "libelle_departement", "annee", "substance", "danger_substance"],
        as_index=False,
    )
    .agg(quantite_kg=("valeur", "sum"))
)

print("Lignes avant / après agrégation :", len(df), "/", len(df_agrege))
print("Départements :", df_agrege["geocode_departement"].nunique(),
      "| années :", df_agrege["annee"].nunique(),
      "| substances :", df_agrege["substance"].nunique())

<a id="section-4"></a>

## 4. Vue d'ensemble : cartes et évolution annuelle

In [ ]:
quantites_departements = (
    df_agrege.groupby(["geocode_departement", "libelle_departement"], as_index=False)
    .agg(quantite_kg=("quantite_kg", "sum"))
    .sort_values("quantite_kg", ascending=False)
)
quantites_departements.head(10)

In [ ]:
# Contours des départements (une seule fois, réutilisés par toutes les cartes)
url_geojson = (
    "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements.geojson"
)
departements = gpd.read_file(url_geojson).rename(columns={"code": "code_dep"})

In [ ]:
carte = departements.merge(
    quantites_departements.rename(columns={"geocode_departement": "code_dep"})[["code_dep", "quantite_kg"]],
    on="code_dep", how="left",
)
print("Départements sans quantité (Corse exclue) :", carte.loc[carte["quantite_kg"].isna(), "nom"].tolist())

fig, ax = plt.subplots(figsize=(10, 9))
carte.plot(
    column="quantite_kg",
    cmap="YlOrRd",
    norm=LogNorm(vmin=carte["quantite_kg"].min(), vmax=carte["quantite_kg"].max()),
    linewidth=0.4,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "#e5e5e5"},
    legend_kwds={"label": "kg cumulés 2013-2023 (échelle log)", "shrink": 0.6},
)
ax.set_title("Achats de produits phytosanitaires par département, 2013-2023", fontsize=14)
ax.set_axis_off()
plt.show()

L'échelle logarithmique est nécessaire : avec une échelle linéaire, quelques grands départements
agricoles écrasent toute la carte.

In [ ]:
quantites_annuelles = df_agrege.groupby("annee")["quantite_kg"].sum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(quantites_annuelles.index, quantites_annuelles.values, marker="o", linewidth=2)
ax.yaxis.set_major_formatter(en_tonnes)
ax.set_xticks(list(ANNEES))
ax.set_title("Achats annuels, toutes substances (France métropolitaine hors Corse)")
ax.set_xlabel("Année")
ax.set_ylabel("Tonnes de substance active")
plt.show()

**Lecture** : le niveau est stable autour de 60 000 à 70 000 t par an à partir de 2014. Le point bas de 2013
vient de données incomplètes, pas d'une vraie baisse. Le pic de 2018 suivi du creux de 2019 correspond
probablement à des achats anticipés avant la hausse de la redevance pour pollutions diffuses au 1er janvier 2019.

**Où les achats ont-ils augmenté ou baissé ?** Pour lisser le pic de 2018 et le creux de 2019,
on compare les moyennes annuelles de **2014-2016** et de **2021-2023**.

In [ ]:
def evolution_entre_periodes(donnees, groupes):
    """Taux d'évolution (%) entre la moyenne annuelle 2014-2016 et celle de 2021-2023."""
    annuel = donnees.groupby([*groupes, "annee"])["quantite_kg"].sum()
    annees = annuel.index.get_level_values("annee")
    debut = annuel[annees.isin([2014, 2015, 2016])].groupby(groupes).sum() / 3
    fin = annuel[annees.isin([2021, 2022, 2023])].groupby(groupes).sum() / 3
    return ((fin / debut.replace(0, np.nan) - 1) * 100).rename("evolution_pct").reset_index()


evolution_dep = evolution_entre_periodes(df_agrege, ["geocode_departement"]).rename(
    columns={"geocode_departement": "code_dep"}
)

fig, ax = plt.subplots(figsize=(10, 9))
tracer_carte(
    departements, evolution_dep, "evolution_pct",
    "Évolution des achats, toutes substances (%)\nmoyenne 2021-2023 vs moyenne 2014-2016",
    ax, cmap="RdBu_r", centrer=True, cle="code_dep",
)
plt.show()

evolution_dep.merge(departements[["code_dep", "nom"]]).sort_values("evolution_pct").iloc[[0, 1, 2, -3, -2, -1]]

<a id="section-5"></a>

## 5. Les 10 molécules du projet

On reprend les 10 molécules choisies pour leur impact sur les cours d'eau (notebook
`tf_selection_molecules_eau`), plutôt qu'un tirage au hasard.

In [ ]:
df_molecules = df_agrege[df_agrege["substance"].isin(MOLECULES)].copy()
manquantes = set(MOLECULES) - set(df_molecules["substance"])
print("Molécules absentes des données :", manquantes or "aucune")

# Une ligne par année et molécule
df_evolution = df_molecules.groupby(["annee", "substance"], as_index=False)["quantite_kg"].sum()
evolution_large = df_evolution.pivot(index="annee", columns="substance", values="quantite_kg").fillna(0)

# Ordre des molécules : de la plus achetée à la moins achetée
ordre = evolution_large.sum().sort_values(ascending=False).index.tolist()
evolution_large = evolution_large[ordre]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.6, 1]})

evolution_large.plot(kind="area", stacked=True, alpha=0.85, ax=axes[0], legend=False)
axes[0].yaxis.set_major_formatter(en_tonnes)
axes[0].set_title("Achats annuels empilés")
axes[0].set_xlabel("Année")
axes[0].set_ylabel("Tonnes")

evolution_large.sum().sort_values().plot(kind="barh", logx=True, color="steelblue", ax=axes[1])
axes[1].set_title("Total 2013-2023 (échelle log)")
axes[1].set_xlabel("kg")
axes[1].set_ylabel("")

fig.legend(*axes[0].get_legend_handles_labels(), title="Substance",
           loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    np.log10(evolution_large.T + 1),
    cmap="YlOrRd",
    linewidths=0.3,
    cbar_kws={"label": "log10(kg + 1)"},
    ax=ax,
)
ax.set_title("Achats par molécule et par année (échelle log)")
ax.set_xlabel("Année")
ax.set_ylabel("")
plt.show()

### Cartes par molécule

Achats cumulés 2013-2023, chaque carte avec sa propre échelle (log), pour voir **où** chaque molécule est achetée.

In [ ]:
cumul_dep_molecule = (
    df_molecules.groupby(["geocode_departement", "substance"], as_index=False)["quantite_kg"].sum()
    .rename(columns={"geocode_departement": "code_dep"})
)
cumul_dep_molecule["log_kg"] = np.log10(cumul_dep_molecule["quantite_kg"] + 1)

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for ax, substance in zip(axes.flat, ordre):
    tracer_carte(
        departements, cumul_dep_molecule[cumul_dep_molecule["substance"] == substance],
        "log_kg", substance, ax, cmap="YlOrRd", cle="code_dep",
    )
fig.suptitle("Achats cumulés 2013-2023 par département, log10(kg + 1)", fontsize=16)
fig.tight_layout()
plt.show()

**Molécule dominante.** Le glyphosate est la plus achetée presque partout. On regarde donc aussi la
molécule la plus achetée **hors glyphosate**, qui renseigne mieux sur le type d'agriculture du département.

In [ ]:
def carte_dominante(donnees, ax, titre):
    """Colorie chaque département selon la molécule la plus achetée."""
    dominante = donnees.loc[donnees.groupby("code_dep")["quantite_kg"].idxmax(), ["code_dep", "substance"]]
    carte = departements.merge(dominante, on="code_dep", how="left")
    carte.plot(
        column="substance", categorical=True, cmap="tab10", legend=True, ax=ax,
        edgecolor="white", linewidth=0.3, missing_kwds={"color": "#e5e5e5", "label": "Corse (exclue)"},
        legend_kwds={"loc": "lower left", "fontsize": 9},
    )
    ax.set_title(titre)
    ax.set_axis_off()


fig, axes = plt.subplots(1, 2, figsize=(18, 8))
carte_dominante(cumul_dep_molecule, axes[0], "Molécule la plus achetée (2013-2023)")
carte_dominante(
    cumul_dep_molecule[cumul_dep_molecule["substance"] != "glyphosate"], axes[1],
    "Molécule la plus achetée hors glyphosate",
)
fig.tight_layout()
plt.show()

Chaque molécule avec sa propre échelle, pour voir les tendances des petites quantités :

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=True)
for ax, substance in zip(axes.flat, ordre):
    ax.plot(evolution_large.index, evolution_large[substance], marker="o")
    ax.yaxis.set_major_formatter(en_tonnes)
    ax.set_title(substance)
    ax.set_ylabel("Tonnes")
    ax.set_ylim(bottom=0)
fig.suptitle("Évolution annuelle des achats par molécule (2013-2023)", fontsize=15)
fig.tight_layout()
plt.show()

In [ ]:
rangs = evolution_large.rank(axis=1, method="min", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
for substance in ordre:
    ax.plot(rangs.index, rangs[substance], marker="o", linewidth=2, label=substance)
ax.invert_yaxis()
ax.set_xticks(list(ANNEES))
ax.set_yticks(range(1, len(MOLECULES) + 1))
ax.set_title("Classement annuel des 10 molécules (1 = la plus achetée)")
ax.set_xlabel("Année")
ax.set_ylabel("Rang")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

In [ ]:
danger = (
    df_molecules.groupby(["annee", "danger_substance"])["quantite_kg"].sum()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 6))
danger.plot(marker="o", linewidth=2, ax=ax)
ax.yaxis.set_major_formatter(en_tonnes)
ax.set_xticks(list(ANNEES))
ax.set_title("Achats des 10 molécules selon la classe de danger")
ax.set_xlabel("Année")
ax.set_ylabel("Tonnes")
ax.legend(title="Classe de danger", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

<a id="section-6"></a>

## 6. Distributions

Une observation = un département et une année (classes de danger additionnées).
Les quantités varient sur plusieurs ordres de grandeur, d'où l'échelle logarithmique.

In [ ]:
obs = df_molecules.groupby(["geocode_departement", "annee", "substance"], as_index=False)["quantite_kg"].sum()
obs["log_kg"] = np.log10(obs["quantite_kg"] + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=obs, x="substance", y="log_kg", order=ordre, color="steelblue", ax=axes[0])
axes[0].set_title("Achats par département et par année")
axes[0].set_xlabel("")
axes[0].set_ylabel("log10(kg + 1)")
axes[0].tick_params(axis="x", rotation=45)
for etiquette in axes[0].get_xticklabels():
    etiquette.set_ha("right")

sns.kdeplot(data=obs, x="log_kg", hue="substance", hue_order=ordre, linewidth=2, ax=axes[1])
axes[1].set_title("Densités (KDE)")
axes[1].set_xlabel("log10(kg + 1)")

fig.tight_layout()
plt.show()

<a id="section-7"></a>

## 7. Concentration géographique

Indice de Gini des achats cumulés entre départements (fonction `calculer_gini` du projet) :
0 = achats répartis également, proche de 1 = concentrés dans quelques départements.

In [ ]:
cumul_departements = (
    df_molecules.groupby(["geocode_departement", "substance"])["quantite_kg"].sum()
    .unstack(fill_value=0)[ordre]
)

indices_gini = cumul_departements.apply(calculer_gini).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

indices_gini.plot(kind="barh", color="steelblue", ax=axes[0])
axes[0].set_xlim(0, 1)
axes[0].set_title("Indice de Gini par molécule")
axes[0].set_xlabel("Gini")

for substance in ordre:
    quantites = np.sort(cumul_departements[substance].to_numpy())
    part_dep = np.arange(len(quantites) + 1) / len(quantites)
    part_qte = np.insert(np.cumsum(quantites) / quantites.sum(), 0, 0)
    axes[1].plot(part_dep, part_qte, label=substance)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="black", label="Égalité parfaite")
axes[1].set_aspect("equal")
axes[1].set_title("Courbes de Lorenz")
axes[1].set_xlabel("Part cumulée des départements")
axes[1].set_ylabel("Part cumulée des quantités")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()
plt.show()

**Corrélation entre molécules** (sur les cumuls par département) : deux molécules fortement corrélées
sont achetées dans les mêmes départements, ce qui renvoie souvent aux mêmes cultures.
On utilise la corrélation de Spearman (sur les rangs), moins sensible aux très gros départements
que celle de Pearson.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cumul_departements.corr(method="spearman"),
    annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0, square=True, ax=ax,
)
ax.set_title("Corrélation de Spearman entre départements")
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()

<a id="section-8"></a>

## 8. Tendances

In [ ]:
total = evolution_large.sum(axis=1)
periode_fiable = total.loc[ANNEE_DEBUT_FIABLE:]
pente, constante = np.polyfit(periode_fiable.index, periode_fiable.values, 1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(total.index, total.values, marker="o", linewidth=2, label="Achats observés")
ax.plot(periode_fiable.index, pente * periode_fiable.index + constante, linestyle="--", linewidth=2,
        label=f"Tendance linéaire {ANNEE_DEBUT_FIABLE}-2023 ({pente / 1e3:+,.0f} t/an)")
ax.yaxis.set_major_formatter(en_tonnes)
ax.set_xticks(list(ANNEES))
ax.set_title("Achats cumulés des 10 molécules")
ax.set_xlabel("Année")
ax.set_ylabel("Tonnes")
ax.legend()
plt.show()

In [ ]:
# Évolution entre le début et la fin de période, par molécule (2014 plutôt que 2013, incomplète)
debut, fin = evolution_large.loc[ANNEE_DEBUT_FIABLE], evolution_large.loc[2023]
variations = pd.DataFrame({
    f"kg_{ANNEE_DEBUT_FIABLE}": debut,
    "kg_2023": fin,
    "variation_pct": (fin / debut.replace(0, np.nan) - 1) * 100,
    "annee_max": evolution_large.idxmax(),
}).sort_values("variation_pct")
variations

**Lecture** : trois molécules ont pratiquement disparu après leur interdiction (isoproturon, imidaclopride,
chlorpyriphos-éthyl) et le boscalid recule nettement. Le glyphosate reste de loin la première molécule
achetée, et le S-métolachlore remonte fortement en 2023.

**Évolution par département et par molécule** (moyenne 2021-2023 vs 2014-2016), pour les molécules
encore vendues. Les molécules quasi disparues sont retirées : leur baisse est d'environ -100 % partout.

In [ ]:
quasi_disparues = variations.index[variations["variation_pct"] < -95].tolist()
molecules_vendues = [m for m in ordre if m not in quasi_disparues]
print("Retirées :", quasi_disparues)

evolution_dep_molecule = evolution_entre_periodes(
    df_molecules, ["geocode_departement", "substance"]
).rename(columns={"geocode_departement": "code_dep"})
# Plafonné à +/-100 % : sinon quelques départements partis de presque rien écrasent l'échelle
evolution_dep_molecule["evolution_pct"] = evolution_dep_molecule["evolution_pct"].clip(-100, 100)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, substance in zip(axes.flat, molecules_vendues):
    tracer_carte(
        departements, evolution_dep_molecule[evolution_dep_molecule["substance"] == substance],
        "evolution_pct", substance, ax, cmap="RdBu_r", centrer=True, cle="code_dep",
    )
for ax in axes.flat[len(molecules_vendues):]:
    ax.set_axis_off()
fig.suptitle("Évolution des achats (%), moyenne 2021-2023 vs 2014-2016, plafonnée à ±100 % (rouge = hausse, bleu = baisse)", fontsize=16)
fig.tight_layout()
plt.show()

<a id="section-9"></a>

## 9. Conclusion

**Qualité des données.** La base est propre : aucune valeur manquante, aucun doublon exact. Les lignes multiples
d'une même substance correspondent au découpage biocontrôle / hors biocontrôle et s'additionnent.
Seule réserve, mais elle est importante : **2013 est incomplète** (environ 24 000 t contre plus de 60 000 t les
autres années). Elle doit être écartée de toute comparaison dans le temps.

**Niveau et géographie.** Depuis 2014, les achats tournent autour de **60 000 à 70 000 t de substance active par an**
(France métropolitaine hors Corse), sans tendance nette. Le pic de 2018 et le creux de 2019 traduisent
probablement des achats anticipés avant la hausse de la redevance pour pollutions diffuses. Les achats sont très
concentrés : les 10 premiers départements, menés par la Gironde, la Marne et la Somme, représentent environ
**31 %** du total. Ce sont des départements de vigne et de grandes cultures.

**Les 10 molécules du projet.** Elles représentent environ **21 %** des quantités achetées, et le glyphosate
pèse à lui seul plus de la moitié de ce sous-ensemble. Leurs trajectoires sont très différentes :
- les interdictions ont été suivies d'effet : l'**isoproturon**, l'**imidaclopride** et le **chlorpyriphos-éthyl**
  ont pratiquement disparu, et le **boscalid** a reculé d'environ 80 % entre 2014 et 2023 ;
- le **glyphosate** baisse d'environ 20 % entre 2014 et 2023, surtout dans le Sud-Ouest et le Sud-Est,
  alors qu'il progresse dans une partie du Bassin parisien ;
- le **S-métolachlore** augmente d'environ 47 %, surtout dans le nord de la France, avec un saut en 2023
  qui pourrait refléter des achats anticipés avant son retrait du marché.

**Deux profils agricoles.** Le cuivre est la molécule la plus concentrée (Gini ≈ 0,73) et la moins corrélée
aux autres : on l'achète dans les zones de vigne. À l'inverse, le glyphosate est la plus diffuse (Gini ≈ 0,40).
Il est fortement corrélé au tébuconazole, au boscalid et à l'imidaclopride (Spearman ≈ 0,9), qui sont des
molécules de grandes cultures. La carte de la molécule dominante hors glyphosate rend cette opposition très
visible.

**Limites.** Ces données mesurent des **achats**, pas des épandages : un produit est compté dans le département
de l'acheteur, pas forcément là où il est utilisé, et l'année d'achat n'est pas forcément l'année d'usage.
Les quantités ne sont pas non plus rapportées à la surface agricole, ce qui avantage les grands départements.
C'est pourquoi la suite du projet répartit ces achats sur les parcelles du RPG puis les agrège par zone
hydrographique (`src/zones_hydrographiques/`) avant de les croiser avec la qualité de l'eau.